# 01 — Data exploration (Phase 2)

Run **ingest** first so `data/processed/` has a file:

```bash
python -m quant_forex ingest --config configs/default.yaml
python -m quant_forex explore --config configs/default.yaml
```

The bundled CSV is **synthetic**. Replace it with real history before any empirical claim.

Close-to-close returns here are descriptive. They are not a strategy and not a backtest.

In [ ]:
from quant_forex.config.loader import load_config
from quant_forex.exploration.io import latest_processed_csv, load_processed_ohlcv
from quant_forex.exploration.pipeline import run_exploration
from quant_forex.exploration.prepare import add_descriptive_columns
from quant_forex.exploration.stats import build_exploration_summary
from quant_forex.utils.paths import ProjectPaths

paths = ProjectPaths.find()
config = load_config(paths.resolve("configs/default.yaml"))
processed = latest_processed_csv(paths.data_processed)
print(processed)
ohlcv = load_processed_ohlcv(processed)
ohlcv.head()

In [ ]:
described = add_descriptive_columns(ohlcv)
described[["timestamp", "close", "simple_return", "log_return", "day_of_week"]].head()

In [ ]:
summary = build_exploration_summary(
    described,
    symbol=str(described["symbol"].iloc[0]) if "symbol" in described.columns else config.instrument.symbol,
    timeframe=config.timeframe,
    expected_frequency=config.data.expected_frequency,
    autocorr_lags=config.exploration.autocorrelation_lags,
)
print(summary.model_dump_json(indent=2)[:1500])
print("sessions_applicable", summary.sessions_applicable)

In [ ]:
result = run_exploration(config, paths, input_csv=processed)
print(result.report_path)
print(result.figure_dir)
list(result.figure_dir.glob("*"))